<a href="https://colab.research.google.com/github/class177/Exercise_2.4/blob/main/HW2_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# -*- coding: utf-8 -*-
"""
Exercise 2.4(b): MIMO Channel GAN Implementation (Colab-ready)

此程式用 Conditional WGAN-GP 來學習 MIMO 通道的條件分佈：
    y = H x + n
條件為：
    condition = [Re(x), Im(x), Re(H), Im(H)]

需要的輸入檔：
    mimo_channel_dataset.mat
其中包含：
    h_mimo : shape [Nr, Nt, Nsnap]，例如 [2, 4, 20000]
"""

import os
import numpy as np
import scipy.io as sio
import matplotlib

# 在 Colab 不需要開視窗，直接存檔即可
matplotlib.use("Agg")
import matplotlib.pyplot as plt

import tensorflow.compat.v1 as tf
tf.disable_v2_behavior()

# =============================================================================
# Configuration
# =============================================================================

# 在 Colab 不綁定實體 GPU 編號，讓系統自動選擇
# 如果你真的要指定 GPU，可以解除註解：
# os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
# os.environ["CUDA_VISIBLE_DEVICES"] = "0"

tf.set_random_seed(100)
np.random.seed(100)

# 檔案與路徑設定
MAT_FILE_PATH   = "mimo_channel_dataset.mat"
MODEL_NAME      = "ChannelGAN_MIMO"
SAVE_FIG_PATH   = MODEL_NAME + "_images"
SAVE_MODEL_PATH = "./Models"

# 訓練與資料設定
DATA_SIZE   = 10000      # 從通道資料生成多少 (x, H, y) 樣本
BATCH_SIZE  = 512
Z_DIM       = 16
TRAIN_ITERS = 10000      # 訓練迭代次數，可依 Colab 時間調整
D_STEPS     = 10         # 每次更新 G 前，更新 D 的次數
PLOT_EVERY  = 1000       # 每多少 iter 存模型與畫圖
NOISE_VAR   = 0.01       # AWGN 每實 / 虛部變異數

# MIMO 天線數（要跟 h_mimo 一致）
NUM_RX = 2
NUM_TX = 4

# 網路維度
OUTPUT_DIM    = 2 * NUM_RX                          # [Re(y1..yNr), Im(y1..yNr)]
CONDITION_DIM = 2 * NUM_TX + 2 * NUM_RX * NUM_TX    # Re/Im(x) + Re/Im(H_flat)

# 16-QAM 星座點
MEAN_SET_QAM = np.asarray([
    -3 - 3j, -3 - 1j, -3 + 1j, -3 + 3j,
    -1 - 3j, -1 - 1j, -1 + 1j, -1 + 3j,
     1 - 3j,  1 - 1j,  1 + 1j,  1 + 3j,
     3 - 3j,  3 - 1j,  3 + 1j,  3 + 3j
], dtype=np.complex64)


# =============================================================================
# Utility functions
# =============================================================================

def sample_Z(sample_size):
    """從 N(0,1) 取樣 GAN 噪音輸入。"""
    return np.random.normal(size=sample_size).astype(np.float32)


def xavier_init(size):
    """Xavier 初始化。"""
    in_dim = size[0]
    xavier_stddev = 1.0 / tf.sqrt(in_dim / 2.0)
    return tf.random_normal(shape=size, stddev=xavier_stddev)


def ensure_dir(path):
    """若資料夾不存在則建立。"""
    if not os.path.exists(path):
        os.makedirs(path)


# =============================================================================
# Data preparation
# =============================================================================

def load_mimo_dataset(mat_file_path):
    """
    從 .mat 檔載入 h_mimo.

    回傳:
        h_dataset: complex ndarray, shape = [Nr, Nt, Nsnap]
    """
    if not os.path.exists(mat_file_path):
        raise FileNotFoundError(
            "找不到 '{}'. 請先把 mimo_channel_dataset.mat 上傳到 Colab 目前目錄。".format(
                mat_file_path
            )
        )

    mat_data = sio.loadmat(mat_file_path)

    if "h_mimo" not in mat_data:
        raise KeyError(
            "檔案 '{}' 中沒有變數 'h_mimo'. 請確認 MATLAB 存檔的變數名稱。".format(
                mat_file_path
            )
        )

    h_dataset = mat_data["h_mimo"]

    if h_dataset.ndim != 3:
        raise ValueError(
            "預期 h_mimo 維度為 [Nr, Nt, Nsnap]，實際為 {}.".format(h_dataset.shape)
        )

    return h_dataset


def generate_mimo_real_samples(h_dataset, number=100, noise_var=0.01):
    """
    根據 MIMO 通道資料產生 (y, condition) 樣本，作為 GAN 的真實資料。

    步驟：
        1. 隨機抽一個通道快照 H (Nr x Nt)
        2. 隨機產生 16QAM 發射向量 x (Nt x 1)
        3. 模擬 y = H x + n
        4. real sample :  [Re(y), Im(y)]
        5. condition   :  [Re(x), Im(x), Re(H_flat), Im(H_flat)] / 3

    Args:
        h_dataset: shape [Nr, Nt, Nsnap] 的複數 ndarray
        number   : 要產生幾個樣本
        noise_var: AWGN 變異數（每實 / 虛部）

    Returns:
        received_data: shape [number, 2*Nr]
        conditioning : shape [number, CONDITION_DIM]
    """
    Nr, Nt, Nsnap = h_dataset.shape

    received_data_list = []
    conditioning_list  = []

    for _ in range(number):
        # 1) 隨機抽通道快照 H
        idx = np.random.choice(Nsnap)
        H = h_dataset[:, :, idx]  # [Nr, Nt], complex

        # 2) 每個 Tx 天線選一個 16QAM symbol
        sym_idx = np.random.choice(len(MEAN_SET_QAM), Nt)
        x = MEAN_SET_QAM[sym_idx].reshape(Nt, 1)  # [Nt, 1]

        # 3) 模擬 y = H x + n
        noise = np.sqrt(noise_var / 2.0) * (
            np.random.randn(Nr, 1) + 1j * np.random.randn(Nr, 1)
        )
        y = H @ x + noise  # [Nr, 1]

        # 4) real sample: [Re(y1..yNr), Im(y1..yNr)]
        y_vec = np.hstack([
            np.real(y).flatten(),
            np.imag(y).flatten()
        ]).astype(np.float32)

        # 5) condition: [Re(x), Im(x), Re(H_flat), Im(H_flat)] / 3
        H_flat = H.flatten()
        cond_vec = np.hstack([
            np.real(x).flatten(),
            np.imag(x).flatten(),
            np.real(H_flat).flatten(),
            np.imag(H_flat).flatten()
        ]).astype(np.float32) / 3.0

        received_data_list.append(y_vec)
        conditioning_list.append(cond_vec)

    received_data = np.asarray(received_data_list, dtype=np.float32)
    conditioning  = np.asarray(conditioning_list,  dtype=np.float32)

    return received_data, conditioning


# =============================================================================
# Model definition (Generator / Discriminator)
# =============================================================================

def generator_conditional(z, conditioning):
    """條件式 Generator：輸入 [z, condition]，輸出 y_hat。"""
    z_combine = tf.concat([z, conditioning], axis=1)
    G_h1 = tf.nn.relu(tf.matmul(z_combine, G_W1) + G_b1)
    G_h2 = tf.nn.relu(tf.matmul(G_h1,      G_W2) + G_b2)
    G_h3 = tf.nn.relu(tf.matmul(G_h2,      G_W3) + G_b3)
    G_logit = tf.matmul(G_h3, G_W4) + G_b4
    return G_logit  # [batch, OUTPUT_DIM]


def discriminator_conditional(X, conditioning):
    """條件式 Discriminator：輸入 [y(or y_hat), condition]，輸出實樣本機率。"""
    x_combine = tf.concat([X, conditioning], axis=1)
    D_h1 = tf.nn.relu(tf.matmul(x_combine / 4.0, D_W1) + D_b1)
    D_h2 = tf.nn.relu(tf.matmul(D_h1,            D_W2) + D_b2)
    D_h3 = tf.nn.relu(tf.matmul(D_h2,            D_W3) + D_b3)
    D_logit = tf.matmul(D_h3, D_W4) + D_b4
    D_prob  = tf.nn.sigmoid(D_logit)
    return D_prob, D_logit


# =============================================================================
# Build computation graph
# =============================================================================

# Discriminator 參數
D_W1 = tf.Variable(xavier_init([OUTPUT_DIM + CONDITION_DIM, 32]))
D_b1 = tf.Variable(tf.zeros(shape=[32]))
D_W2 = tf.Variable(xavier_init([32, 32]))
D_b2 = tf.Variable(tf.zeros(shape=[32]))
D_W3 = tf.Variable(xavier_init([32, 32]))
D_b3 = tf.Variable(tf.zeros(shape=[32]))
D_W4 = tf.Variable(xavier_init([32, 1]))
D_b4 = tf.Variable(tf.zeros(shape=[1]))
theta_D = [D_W1, D_W2, D_W3, D_b1, D_b2, D_b3, D_W4, D_b4]

# Generator 參數
G_W1 = tf.Variable(xavier_init([Z_DIM + CONDITION_DIM, 128]))
G_b1 = tf.Variable(tf.zeros(shape=[128]))
G_W2 = tf.Variable(xavier_init([128, 128]))
G_b2 = tf.Variable(tf.zeros(shape=[128]))
G_W3 = tf.Variable(xavier_init([128, 128]))
G_b3 = tf.Variable(tf.zeros(shape=[128]))
G_W4 = tf.Variable(xavier_init([128, OUTPUT_DIM]))
G_b4 = tf.Variable(tf.zeros(shape=[OUTPUT_DIM]))
theta_G = [G_W1, G_W2, G_W3, G_b1, G_b2, G_b3, G_W4, G_b4]

# Placeholders
R_sample  = tf.placeholder(tf.float32, shape=[None, OUTPUT_DIM])
Z         = tf.placeholder(tf.float32, shape=[None, Z_DIM])
Condition = tf.placeholder(tf.float32, shape=[None, CONDITION_DIM])

# Forward pass
G_sample       = generator_conditional(Z, Condition)
D_prob_real, D_logit_real = discriminator_conditional(R_sample,  Condition)
D_prob_fake, D_logit_fake = discriminator_conditional(G_sample, Condition)

# WGAN-GP 損失
D_loss = tf.reduce_mean(D_logit_fake) - tf.reduce_mean(D_logit_real)
G_loss = -tf.reduce_mean(D_logit_fake)

lambda_gp = 5.0
alpha = tf.random_uniform(shape=tf.shape(R_sample), minval=0.0, maxval=1.0)
differences  = G_sample - R_sample
interpolates = R_sample + alpha * differences
_, D_inter = discriminator_conditional(interpolates, Condition)
gradients = tf.gradients(D_inter, [interpolates])[0]
slopes = tf.sqrt(tf.reduce_sum(tf.square(gradients), axis=1) + 1e-8)
gradient_penalty = tf.reduce_mean((slopes - 1.0) ** 2)
D_loss += lambda_gp * gradient_penalty

D_solver = tf.train.AdamOptimizer(
    learning_rate=1e-4, beta1=0.5, beta2=0.9
).minimize(D_loss, var_list=theta_D)

G_solver = tf.train.AdamOptimizer(
    learning_rate=1e-4, beta1=0.5, beta2=0.9
).minimize(G_loss, var_list=theta_G)


# =============================================================================
# Plot helpers
# =============================================================================

def plot_real_distribution(data, save_fig_path):
    """
    畫 Rx 第 1 根天線的實際分佈：
        x 軸：Re(y1)
        y 軸：Im(y1)
    """
    plt.figure(figsize=(5, 5))
    plt.plot(data[:1000, 0], data[:1000, NUM_RX], "b.", alpha=0.6)
    plt.xlabel(r"$Re\{y_1\}$")
    plt.ylabel(r"$Im\{y_1\}$")
    plt.title("Real data distribution (Rx antenna 1)")
    plt.grid(True, alpha=0.3)
    plt.savefig(os.path.join(save_fig_path, "real_rx1.png"), bbox_inches="tight")
    plt.close()


def build_plot_conditioning_from_fixed_channel(H, number=20):
    """
    為了可視化，固定一個 MIMO 通道 H，只掃 Tx1 的 16QAM 星座，
    其他 Tx 設為 0，重複 number 次。
    回傳 conditioning: shape [16*number, CONDITION_DIM]
    """
    H_flat = H.flatten()
    cond_list = []

    for qam_symbol in MEAN_SET_QAM:
        for _ in range(number):
            x = np.zeros((NUM_TX, 1), dtype=np.complex64)
            x[0, 0] = qam_symbol

            cond_vec = np.hstack([
                np.real(x).flatten(),
                np.imag(x).flatten(),
                np.real(H_flat).flatten(),
                np.imag(H_flat).flatten()
            ]).astype(np.float32) / 3.0

            cond_list.append(cond_vec)

    return np.asarray(cond_list, dtype=np.float32)


def plot_generated_samples(sess, h_dataset, step, save_fig_path):
    """
    固定隨機一個 H，畫出 Generator 產生的 Rx1 複數平面分佈。
    """
    idx = np.random.choice(h_dataset.shape[2])
    H = h_dataset[:, :, idx]

    conditioning = build_plot_conditioning_from_fixed_channel(H, number=20)
    z_input = sample_Z((conditioning.shape[0], Z_DIM))
    samples = sess.run(
        G_sample,
        feed_dict={Z: z_input, Condition: conditioning}
    )

    plt.figure(figsize=(5, 5))
    plt.plot(samples[:, 0], samples[:, NUM_RX], "r.", alpha=0.6)
    plt.xlabel(r"$Re\{\hat{y}_1\}$")
    plt.ylabel(r"$Im\{\hat{y}_1\}$")
    plt.title("Generated samples (Rx antenna 1) at step {}".format(step))
    plt.grid(True, alpha=0.3)
    plt.savefig(
        os.path.join(save_fig_path, "generated_rx1_step_{:06d}.png".format(step)),
        bbox_inches="tight"
    )
    plt.close()


# =============================================================================
# Main training loop
# =============================================================================

def main():
    ensure_dir(SAVE_FIG_PATH)
    ensure_dir(SAVE_MODEL_PATH)

    # 1. 載入 MIMO 通道資料
    h_dataset = load_mimo_dataset(MAT_FILE_PATH)
    print("Loaded h_dataset shape =", h_dataset.shape)

    if h_dataset.shape[0] != NUM_RX or h_dataset.shape[1] != NUM_TX:
        raise ValueError(
            "預期 h_dataset 前兩維為 ({}, {}), 實際為 {}.".format(
                NUM_RX, NUM_TX, h_dataset.shape
            )
        )

    # 2. 產生訓練樣本
    data, cond_all = generate_mimo_real_samples(
        h_dataset, number=DATA_SIZE, noise_var=NOISE_VAR
    )
    print("data shape        =", data.shape)
    print("conditioning shape =", cond_all.shape)

    # 畫真實資料分佈
    plot_real_distribution(data, SAVE_FIG_PATH)

    # 3. 開始訓練
    sess = tf.Session()
    sess.run(tf.global_variables_initializer())
    saver = tf.train.Saver()

    for it in range(TRAIN_ITERS):
        start_idx = (it * BATCH_SIZE) % DATA_SIZE

        if start_idx + BATCH_SIZE >= len(data):
            # 走完一輪資料就重新洗牌
            perm = np.random.permutation(DATA_SIZE)
            data    = data[perm]
            cond_all = cond_all[perm]
            start_idx = 0

        X_mb   = data[start_idx:start_idx + BATCH_SIZE, :]
        cond_mb = cond_all[start_idx:start_idx + BATCH_SIZE, :]

        # 更新 Discriminator
        for _ in range(D_STEPS):
            _, D_loss_curr = sess.run(
                [D_solver, D_loss],
                feed_dict={
                    R_sample: X_mb,
                    Z: sample_Z((BATCH_SIZE, Z_DIM)),
                    Condition: cond_mb
                }
            )

        # 更新 Generator
        _, G_loss_curr = sess.run(
            [G_solver, G_loss],
            feed_dict={
                R_sample: X_mb,
                Z: sample_Z((BATCH_SIZE, Z_DIM)),
                Condition: cond_mb
            }
        )

        if (it + 1) % 100 == 0:
            print("Iter: {:6d} | D_loss: {:8.4f} | G_loss: {:8.4f}".format(
                it + 1, D_loss_curr, G_loss_curr
            ))

        if (it + 1) % PLOT_EVERY == 0:
            ckpt_path = saver.save(
                sess,
                os.path.join(SAVE_MODEL_PATH,
                             "ChannelGAN_MIMO_step_{}.ckpt".format(it + 1))
            )
            print("Model saved to:", ckpt_path)

            plot_generated_samples(sess, h_dataset, it + 1, SAVE_FIG_PATH)

    # 最後存一次模型
    ckpt_path = saver.save(
        sess,
        os.path.join(SAVE_MODEL_PATH, "ChannelGAN_MIMO_final.ckpt")
    )
    print("Final model saved to:", ckpt_path)

    sess.close()


if __name__ == "__main__":
    main()


Loaded h_dataset shape = (2, 4, 20000)
data shape        = (10000, 4)
conditioning shape = (10000, 24)
Iter:    100 | D_loss:   1.7816 | G_loss:  -0.9656
Iter:    200 | D_loss:   1.2791 | G_loss:  -0.3240
Iter:    300 | D_loss:  -0.5191 | G_loss:   0.7665
Iter:    400 | D_loss:   1.2809 | G_loss:  -0.2930
Iter:    500 | D_loss:   0.0227 | G_loss:   0.5791
Iter:    600 | D_loss:  -0.1493 | G_loss:   0.2786
Iter:    700 | D_loss:  -0.1214 | G_loss:   0.3006
Iter:    800 | D_loss:  -0.0760 | G_loss:   0.2091
Iter:    900 | D_loss:  -0.0247 | G_loss:   0.1679
Iter:   1000 | D_loss:   0.0247 | G_loss:   0.1244
Model saved to: ./Models/ChannelGAN_MIMO_step_1000.ckpt
Iter:   1100 | D_loss:   0.0308 | G_loss:   0.2234
Iter:   1200 | D_loss:   0.0401 | G_loss:  -0.2089
Iter:   1300 | D_loss:   0.0273 | G_loss:  -0.4249
Iter:   1400 | D_loss:   0.0405 | G_loss:  -0.5876
Iter:   1500 | D_loss:   0.0247 | G_loss:  -0.4846
Iter:   1600 | D_loss:   0.0664 | G_loss:  -0.4558
Iter:   1700 | D_loss:   

In [1]:
# -*- coding: utf-8 -*-
"""
Exercise 2.4: Channel GAN Implementation (Colab-ready)

This script simulates a Rayleigh channel using a Conditional Generative
Adversarial Network (CGAN / WGAN-GP style). It learns the channel-output
distribution y = h x + n, conditioned on the transmitted symbol x and
the channel coefficient h, without explicit CSI estimation.
"""

import os
import numpy as np
import scipy.io as sio
import matplotlib
matplotlib.use('Agg')  # 背景繪圖，不彈出視窗
import matplotlib.pyplot as plt

import tensorflow.compat.v1 as tf
tf.disable_v2_behavior()

# 固定隨機種子（方便重現結果）
tf.set_random_seed(100)
np.random.seed(100)

# ======================= Generator & Discriminator ======================= #
def generator_conditional(z, conditioning):
    """
    Generator for CGAN.

    Args:
        z           : Noise vector (batch_size, Z_dim)
        conditioning: Conditioning vector (batch_size, condition_dim)

    Returns:
        G_logit     : Generated received samples (batch_size, 2)
    """
    z_combine = tf.concat([z, conditioning], axis=1)
    G_h1 = tf.nn.relu(tf.matmul(z_combine, G_W1) + G_b1)
    G_h2 = tf.nn.relu(tf.matmul(G_h1, G_W2) + G_b2)
    G_h3 = tf.nn.relu(tf.matmul(G_h2, G_W3) + G_b3)
    G_logit = tf.matmul(G_h3, G_W4) + G_b4
    return G_logit


def discriminator_conditional(X, conditioning):
    """
    Discriminator for CGAN.

    Args:
        X           : Input sample (real or generated), shape (batch_size, 2)
        conditioning: Conditioning vector, shape (batch_size, condition_dim)

    Returns:
        D_prob      : Probability that input is real (after sigmoid)
        D_logit     : Logit output (before sigmoid)
    """
    z_combine = tf.concat([X, conditioning], axis=1)
    D_h1_real = tf.nn.relu(tf.matmul(z_combine / 4.0, D_W1) + D_b1)
    D_h2_real = tf.nn.relu(tf.matmul(D_h1_real, D_W2) + D_b2)
    D_h3_real = tf.nn.relu(tf.matmul(D_h2_real, D_W3) + D_b3)
    D_logit = tf.matmul(D_h3_real, D_W4) + D_b4
    D_prob = tf.nn.sigmoid(D_logit)
    return D_prob, D_logit


# =========================== Utility Functions =========================== #
def sample_Z(sample_size):
    """Sampling generation noise Z from standard normal distribution."""
    return np.random.normal(size=sample_size).astype(np.float32)


def xavier_init(size):
    """Xavier initialization for weight matrices."""
    in_dim = size[0]
    xavier_stddev = 1. / tf.sqrt(in_dim / 2.)
    return tf.random_normal(shape=size, stddev=xavier_stddev)


# =================== Data Preparation: Rayleigh Channel ================== #
def generate_real_samples_with_labels_Rayleigh(h_dataset, number=100,
                                               mean_set_QAM=None,
                                               noise_var=0.03):
    """
    Generate real (labeled) samples for training the CGAN.

    Steps:
        1. Randomly select channel coefficients h from h_dataset.
        2. Randomly generate 16QAM symbols x.
        3. Simulate y = h * x + n, with complex Gaussian noise n.
        4. Construct the conditioning vector:
           conditioning = [Re{x}, Im{x}, Re{h}, Im{h}] / 3

    Args:
        h_dataset   : 1D numpy array of complex channel coefficients (Rayleigh).
        number      : Number of samples to generate.
        mean_set_QAM: Constellation points for QAM (complex ndarray of len M).
        noise_var   : Noise variance for each real/imag component.

    Returns:
        received_data: ndarray of shape (number, 2) -> [Re{y}, Im{y}]
        conditioning : ndarray of shape (number, 4) -> [Re{x}, Im{x}, Re{h}, Im{h}] / 3
        one_hot      : ndarray of shape (number, M), one-hot label of chosen symbol index
    """
    if mean_set_QAM is None:
        raise ValueError("mean_set_QAM must be provided.")

    M = len(mean_set_QAM)  # constellation size (e.g., 16 for 16QAM)

    # 1) 隨機抽通道係數 h
    h_indices = np.random.randint(0, len(h_dataset), size=number)
    h_complex = h_dataset[h_indices].astype(np.complex64)

    # 2) 隨機選 16QAM 符號
    symbol_indices = np.random.randint(0, M, size=number)
    x_complex = mean_set_QAM[symbol_indices].astype(np.complex64)

    # 3) 計算 y = h * x + n
    y_clean = h_complex * x_complex

    # 產生複數高斯雜訊 n = n_r + j n_i
    cov = [[noise_var, 0.0], [0.0, noise_var]]
    noise = np.random.multivariate_normal([0.0, 0.0], cov, size=number).astype(np.float32)
    n_complex = noise[:, 0] + 1j * noise[:, 1]

    y_complex = y_clean + n_complex

    # 4) 構造 conditioning = [Re{x}, Im{x}, Re{h}, Im{h}] / 3
    x_r = np.real(x_complex).reshape(-1, 1)
    x_i = np.imag(x_complex).reshape(-1, 1)
    h_r = np.real(h_complex).reshape(-1, 1)
    h_i = np.imag(h_complex).reshape(-1, 1)

    conditioning = np.hstack([x_r, x_i, h_r, h_i]).astype(np.float32) / 3.0

    # 接收資料 [Re{y}, Im{y}]
    y_r = np.real(y_complex).reshape(-1, 1).astype(np.float32)
    y_i = np.imag(y_complex).reshape(-1, 1).astype(np.float32)
    received_data = np.hstack([y_r, y_i])

    # one-hot label for constellation index（如果你後面要用）
    one_hot = np.zeros((number, M), dtype=np.float32)
    one_hot[np.arange(number), symbol_indices] = 1.0

    return received_data, conditioning, one_hot


# =============================== Main ==================================== #
# 16QAM 星座點
mean_set_QAM = np.asarray(
    [-3 - 3j, -3 - 1j, -3 + 1j, -3 + 3j,
     -1 - 3j, -1 - 1j, -1 + 1j, -1 + 3j,
      1 - 3j,  1 - 1j,  1 + 1j,  1 + 3j,
      3 - 3j,  3 - 1j,  3 + 1j,  3 + 3j],
    dtype=np.complex64
)

# 讀取 Rayleigh channel dataset (.mat)
mat_file_path = 'rayleigh_channel_dataset.mat'  # 請先上傳到 Colab 目前目錄

if not os.path.exists(mat_file_path):
    raise FileNotFoundError(
        "找不到 rayleigh_channel_dataset.mat，請先在 Colab 左側 'Files' 面板上傳此檔案。"
    )

mat_data = sio.loadmat(mat_file_path)
h_dataset = mat_data['h_siso'].flatten()
print("Loaded h_dataset with shape:", h_dataset.shape)

# 基本設定
batch_size = 512
condition_dim = 4      # [Re{x}, Im{x}, Re{h}, Im{h}]
Z_dim = 16
model = 'ChannelGAN_Rayleigh_'
data_size = 10000

# 產生真實資料（供 GAN 訓練用）
data, conditioning_all, one_hot_labels = generate_real_samples_with_labels_Rayleigh(
    h_dataset, number=data_size, mean_set_QAM=mean_set_QAM, noise_var=0.03
)

# ============================ Graph Construction ========================= #
# Discriminator 參數
D_W1 = tf.Variable(xavier_init([2 + condition_dim, 32]))
D_b1 = tf.Variable(tf.zeros(shape=[32]))
D_W2 = tf.Variable(xavier_init([32, 32]))
D_b2 = tf.Variable(tf.zeros(shape=[32]))
D_W3 = tf.Variable(xavier_init([32, 32]))
D_b3 = tf.Variable(tf.zeros(shape=[32]))
D_W4 = tf.Variable(xavier_init([32, 1]))
D_b4 = tf.Variable(tf.zeros(shape=[1]))
theta_D = [D_W1, D_W2, D_W3, D_b1, D_b2, D_b3, D_W4, D_b4]

# Generator 參數
G_W1 = tf.Variable(xavier_init([Z_dim + condition_dim, 128]))
G_b1 = tf.Variable(tf.zeros(shape=[128]))
G_W2 = tf.Variable(xavier_init([128, 128]))
G_b2 = tf.Variable(tf.zeros(shape=[128]))
G_W3 = tf.Variable(xavier_init([128, 128]))
G_b3 = tf.Variable(tf.zeros(shape=[128]))
G_W4 = tf.Variable(xavier_init([128, 2]))
G_b4 = tf.Variable(tf.zeros(shape=[2]))
theta_G = [G_W1, G_W2, G_W3, G_b1, G_b2, G_b3, G_W4, G_b4]

# Placeholder
R_sample = tf.placeholder(tf.float32, shape=[None, 2])
Z = tf.placeholder(tf.float32, shape=[None, Z_dim])
Condition = tf.placeholder(tf.float32, shape=[None, condition_dim])

# 建立網路
G_sample = generator_conditional(Z, Condition)
D_prob_real, D_logit_real = discriminator_conditional(R_sample, Condition)
D_prob_fake, D_logit_fake = discriminator_conditional(G_sample, Condition)

# ============================ Loss & Optimizer =========================== #
# WGAN-GP 損失
D_loss = tf.reduce_mean(D_logit_fake) - tf.reduce_mean(D_logit_real)
G_loss = -1.0 * tf.reduce_mean(D_logit_fake)

# Gradient Penalty
lambdda = 5.0
alpha = tf.random_uniform(shape=tf.shape(R_sample), minval=0., maxval=1.)
differences = G_sample - R_sample
interpolates = R_sample + alpha * differences
_, D_inter = discriminator_conditional(interpolates, Condition)
gradients = tf.gradients(D_inter, [interpolates])[0]
slopes = tf.sqrt(tf.reduce_sum(tf.square(gradients), axis=1))
gradient_penalty = tf.reduce_mean((slopes - 1.0) ** 2)
D_loss += lambdda * gradient_penalty

D_solver = tf.train.AdamOptimizer(learning_rate=1e-4, beta1=0.5, beta2=0.9)\
    .minimize(D_loss, var_list=theta_D)
G_solver = tf.train.AdamOptimizer(learning_rate=1e-4, beta1=0.5, beta2=0.9)\
    .minimize(G_loss, var_list=theta_G)

# ============================ Session & Training ========================= #
sess = tf.Session()
sess.run(tf.global_variables_initializer())

save_fig_path = model + "images"
if not os.path.exists(save_fig_path):
    os.makedirs(save_fig_path)

saver = tf.train.Saver()
plot_every = 1000
num_iters = 20000   # Colab 不要先跑 750000，先設小一點避免太久

print("Start training ...")

for it in range(num_iters):
    start_idx = (it * batch_size) % data_size
    if start_idx + batch_size > data_size:
        # 確保 batch 連續
        idx = np.arange(batch_size) % data_size
        X_mb = data[idx]
        Cond_mb = conditioning_all[idx]
    else:
        X_mb = data[start_idx:start_idx + batch_size]
        Cond_mb = conditioning_all[start_idx:start_idx + batch_size]

    # Update D 多次
    for _ in range(5):
        _, D_loss_curr = sess.run(
            [D_solver, D_loss],
            feed_dict={
                R_sample: X_mb,
                Z: sample_Z((batch_size, Z_dim)),
                Condition: Cond_mb
            }
        )

    # Update G 一次
    _, G_loss_curr = sess.run(
        [G_solver, G_loss],
        feed_dict={
            R_sample: X_mb,
            Z: sample_Z((batch_size, Z_dim)),
            Condition: Cond_mb
        }
    )

    if (it + 1) % 500 == 0:
        print("Iter: {:5d}, D_loss: {: .4f}, G_loss: {: .4f}".format(
            it + 1, D_loss_curr, G_loss_curr))

    # 定期畫圖 / 儲存
    if (it + 1) % plot_every == 0:
        save_path = saver.save(sess, './ChannelGAN_model_step_{}.ckpt'.format(it + 1))
        print("Model saved at:", save_path)

        plt.figure(figsize=(5, 5))
        # 抽一些條件，畫出生成樣本分佈
        number = 200
        # 隨機選一個通道，讓星座圖好看
        h_complex_for_plot = np.random.choice(h_dataset, 1)[0]
        h_r_plot = np.real(h_complex_for_plot)
        h_i_plot = np.imag(h_complex_for_plot)

        all_gen = []
        for idx_sym in range(len(mean_set_QAM)):
            idx_vec = np.full(number, idx_sym, dtype=int)
            x_complex_plot = mean_set_QAM[idx_vec]

            x_r = np.real(x_complex_plot).reshape(-1, 1)
            x_i = np.imag(x_complex_plot).reshape(-1, 1)
            h_r = np.full((number, 1), h_r_plot)
            h_i = np.full((number, 1), h_i_plot)
            cond_plot = np.hstack([x_r, x_i, h_r, h_i]).astype(np.float32) / 3.0

            gen_samples = sess.run(
                G_sample,
                feed_dict={
                    Z: sample_Z((number, Z_dim)),
                    Condition: cond_plot
                }
            )
            all_gen.append(gen_samples)
            plt.plot(gen_samples[:, 0], gen_samples[:, 1], 'b.', alpha=0.5)

        plt.xlim([-4, 4])
        plt.ylim([-4, 4])
        plt.xlabel(r'Re{y}')
        plt.ylabel(r'Im{y}')
        plt.title('Iter {}: D_loss={:.2f}, G_loss={:.2f}'.format(
            it + 1, D_loss_curr, G_loss_curr))
        fig_path = os.path.join(save_fig_path, 'scatter_iter_{:06d}.png'.format(it + 1))
        plt.savefig(fig_path, bbox_inches='tight')
        plt.close()
        print("Saved figure to:", fig_path)

print("Training finished.")


Instructions for updating:
non-resource variables are not supported in the long term


Loaded h_dataset with shape: (20000,)
Start training ...
Iter:   500, D_loss:  0.2454, G_loss: -0.6625
Iter:  1000, D_loss:  0.4436, G_loss:  0.1280
Model saved at: ./ChannelGAN_model_step_1000.ckpt
Saved figure to: ChannelGAN_Rayleigh_images/scatter_iter_001000.png
Iter:  1500, D_loss:  0.1986, G_loss:  0.8675
Iter:  2000, D_loss:  0.2151, G_loss:  0.9279
Model saved at: ./ChannelGAN_model_step_2000.ckpt
Saved figure to: ChannelGAN_Rayleigh_images/scatter_iter_002000.png
Iter:  2500, D_loss:  nan, G_loss:  nan
Iter:  3000, D_loss:  nan, G_loss:  nan
Model saved at: ./ChannelGAN_model_step_3000.ckpt
Saved figure to: ChannelGAN_Rayleigh_images/scatter_iter_003000.png
Iter:  3500, D_loss:  nan, G_loss:  nan
Iter:  4000, D_loss:  nan, G_loss:  nan
Model saved at: ./ChannelGAN_model_step_4000.ckpt
Saved figure to: ChannelGAN_Rayleigh_images/scatter_iter_004000.png
Iter:  4500, D_loss:  nan, G_loss:  nan
Iter:  5000, D_loss:  nan, G_loss:  nan
Model saved at: ./ChannelGAN_model_step_5000.ck

Instructions for updating:
Use standard file APIs to delete files with this prefix.


Iter:  6000, D_loss:  nan, G_loss:  nan
Model saved at: ./ChannelGAN_model_step_6000.ckpt
Saved figure to: ChannelGAN_Rayleigh_images/scatter_iter_006000.png
Iter:  6500, D_loss:  nan, G_loss:  nan
Iter:  7000, D_loss:  nan, G_loss:  nan
Model saved at: ./ChannelGAN_model_step_7000.ckpt
Saved figure to: ChannelGAN_Rayleigh_images/scatter_iter_007000.png
Iter:  7500, D_loss:  nan, G_loss:  nan
Iter:  8000, D_loss:  nan, G_loss:  nan
Model saved at: ./ChannelGAN_model_step_8000.ckpt
Saved figure to: ChannelGAN_Rayleigh_images/scatter_iter_008000.png
Iter:  8500, D_loss:  nan, G_loss:  nan
Iter:  9000, D_loss:  nan, G_loss:  nan
Model saved at: ./ChannelGAN_model_step_9000.ckpt
Saved figure to: ChannelGAN_Rayleigh_images/scatter_iter_009000.png
Iter:  9500, D_loss:  nan, G_loss:  nan
Iter: 10000, D_loss:  nan, G_loss:  nan
Model saved at: ./ChannelGAN_model_step_10000.ckpt
Saved figure to: ChannelGAN_Rayleigh_images/scatter_iter_010000.png
Iter: 10500, D_loss:  nan, G_loss:  nan
Iter: 110